In [1]:
!pip install librosa transformers wandb --quiet

In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import wandb

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

2026-03-10 06:34:52.416958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773124492.725725      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773124492.820646      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773124493.588504      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773124493.588565      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773124493.588568      24 computation_placer.cc:177] computation placer alr

Using device: cuda


**Paths and basic config**

In [3]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES_PATH = os.path.join(DATA_ROOT, "genres_stems")
NOISE_PATH  = os.path.join(DATA_ROOT, "ESC-50-master/audio")
MASHUP_DIR  = os.path.join(DATA_ROOT, "mashups")
STEM_NAMES  = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]

SAMPLE_RATE  = 16000   
NUM_SAMPLES  = SAMPLE_RATE * 10  
WANDB_PROJECT = "dl-genai-project-26-t1"


16kHz works for all three models

**Build genre + song index**

In [4]:
genre_names  = sorted([g for g in os.listdir(GENRES_PATH)
                        if os.path.isdir(os.path.join(GENRES_PATH, g))])
genre_to_idx = {g: i for i, g in enumerate(genre_names)}
idx_to_genre = {i: g for g, i in genre_to_idx.items()}
print("Genres found:", genre_names)

all_song_paths, all_labels = [], []
for genre in genre_names:
    gdir  = os.path.join(GENRES_PATH, genre)
    label = genre_to_idx[genre]
    for song in os.listdir(gdir):
        spath = os.path.join(gdir, song)
        if os.path.isdir(spath):
            all_song_paths.append(spath)
            all_labels.append(label)

print("Total songs:", len(all_song_paths))

Genres found: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Total songs: 1000


**Split into train/val — same split used by all 3 models for fair comparison**

In [5]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_song_paths, all_labels,
    test_size=0.15, stratify=all_labels, random_state=42
)

train_songs_by_genre = {i: [] for i in range(len(genre_names))}
for p, l in zip(train_paths, train_labels):
    train_songs_by_genre[l].append(p)

val_songs_by_genre = {i: [] for i in range(len(genre_names))}
for p, l in zip(val_paths, val_labels):
    val_songs_by_genre[l].append(p)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")


Train: 850 | Val: 150


**Load ESC-50 noise clips**

In [6]:
print("Loading noise clips...")
noise_clips = []
for f in sorted(os.listdir(NOISE_PATH)):
    if not f.endswith(".wav"):
        continue
    y, _ = librosa.load(os.path.join(NOISE_PATH, f), sr=SAMPLE_RATE, mono=True)
    noise_clips.append(y.astype(np.float32))
print(f"Loaded {len(noise_clips)} noise clips")

Loading noise clips...
Loaded 2000 noise clips


**Shared audio helper functions**

In [7]:
def load_stem(path, num_samples, random_crop=True):
    if not os.path.exists(path):
        return np.zeros(num_samples, dtype=np.float32)
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(y) < num_samples:
        y = np.pad(y, (0, num_samples - len(y)))
    elif random_crop:
        start = random.randint(0, len(y) - num_samples)
        y = y[start:start + num_samples]
    else:
        y = y[:num_samples]
    return y.astype(np.float32)


def make_mashup(label, songs_by_genre, num_samples):
    pool      = songs_by_genre[label]
    audio_mix = np.zeros(num_samples, dtype=np.float32)
    for stem in STEM_NAMES:
        src        = random.choice(pool)
        y          = load_stem(os.path.join(src, stem), num_samples)
        audio_mix += random.uniform(0.5, 1.5) * y
    return audio_mix


def add_noise(audio, noise_clips):
    noise = random.choice(noise_clips).copy()
    n     = len(audio)
    if len(noise) < n:
        noise = np.tile(noise, int(np.ceil(n / len(noise))))
    start     = random.randint(0, len(noise) - n)
    noise     = noise[start:start + n]
    level     = random.uniform(0.05, 0.3)
    sig_rms   = np.sqrt(np.mean(audio ** 2)) + 1e-9
    noise_rms = np.sqrt(np.mean(noise ** 2)) + 1e-9
    noise     = noise * (sig_rms / noise_rms) * level
    return audio + noise


def audio_to_melspec(audio, n_mels=64):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=n_mels, n_fft=1024, hop_length=512
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    t      = torch.tensor(mel_db).float()
    t      = (t - t.mean()) / (t.std() + 1e-6)
    return t.unsqueeze(0).repeat(3, 1, 1)   


def audio_to_mfcc(audio, n_mfcc=40):
    mfcc      = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=n_mfcc)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std  = np.std(mfcc, axis=1)
    return np.concatenate([mfcc_mean, mfcc_std]) 


load stem -> Load one stem wav file and crop/pad to fixed length.
make mashup -> Mix stems from different songs of the same genre.This matches how the test mashups were created.
add noise -> Add a random ESC-50 noise clip to the audio.
audio to melspec -> Convert audio to a normalised mel spectrogram tensor (3 x H x W).
audio to mfcc -> Extract MFCC features — used by the SVM model. Use mean and std across time as a fixed-size feature vector



In [8]:
ast_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
).to(device)

ast_model.load_state_dict(
    torch.load("/kaggle/input/models/parikshit011/best-ast/pytorch/default/1/best_ast_0.85.pth")
)
ast_model.eval()
print("Model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


**INFERENCE — use best AST model to generate submission**

Load best AST and run inference with TTA

In [9]:
from transformers import ASTFeatureExtractor
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
print("Feature extractor loaded")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Feature extractor loaded


In [10]:
def audio_to_ast_input(audio):
    inputs = feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_tensors="pt"
    )
    return inputs["input_values"].squeeze(0)

In [11]:
TTA_CROPS = 8
CROP_LEN  = SAMPLE_RATE * 10

In [12]:
ast_model.eval()
softmax = nn.Softmax(dim=-1)

TTA_CROPS = 8
CROP_LEN  = SAMPLE_RATE * 10

test_df   = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
all_probs = []
all_ids   = []

print(f"Running inference with TTA ({TTA_CROPS} crops) on {len(test_df)} files")

with torch.no_grad():
    for i, row in test_df.iterrows():
        sample_id = row["id"]
        fname     = row["filename"].split("/")[-1]
        path      = os.path.join(MASHUP_DIR, fname)

        audio, _   = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        n          = len(audio)
        crop_probs = []

        for _ in range(TTA_CROPS):
            if n >= CROP_LEN:
                start = random.randint(0, n - CROP_LEN)
                clip  = audio[start:start + CROP_LEN]
            else:
                clip = np.pad(audio, (0, CROP_LEN - n))

            inp   = audio_to_ast_input(clip).unsqueeze(0).to(device)
            out   = ast_model(input_values=inp)
            probs = softmax(out.logits)
            crop_probs.append(probs.cpu().numpy())

        avg_prob = np.mean(crop_probs, axis=0)
        all_probs.append(avg_prob[0])
        all_ids.append(sample_id)

        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(test_df)} done")

all_probs = np.array(all_probs)
preds     = np.argmax(all_probs, axis=1)

submission = pd.DataFrame({
    "id":    all_ids,
    "genre": [idx_to_genre[p] for p in preds]
})
submission.to_csv("submission.csv", index=False)

print("\nPrediction distribution:")
print(submission["genre"].value_counts())

Running inference with TTA (8 crops) on 3020 files
  500/3020 done
  1000/3020 done
  1500/3020 done
  2000/3020 done
  2500/3020 done
  3000/3020 done

Prediction distribution:
genre
jazz         427
rock         414
pop          333
blues        333
metal        314
disco        299
hiphop       296
reggae       295
country      254
classical     55
Name: count, dtype: int64
